# did_newspaper

> 对应代码：`EconometricsCode/code_in_notes/Chap.15/did_newspaper.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.15`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.15")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


读入 Angelucci & Cagé (AEJ Micro) 的法国报纸数据，研究 1968 年前后全国性报纸与地方报纸广告价格的变化。

In [ ]:
%%stata
clear
use ../datasets/Angelucci_Cage_AEJMicro_dataset.dta, clear


保留 1960–1974 年样本；定义政策后虚拟变量 after（1968 年起）与“处理×事后”交互项 treat_post，其中 national 为全国性报纸（处理组）。

In [ ]:
%%stata
drop if year<1960 | year>1974
gen after = year>1967
gen treat_post = national*after


由于 1960、1961 年缺少价格数据，事件研究只画 1962 年起。先生成一个恒为 1 的变量 p 并标注为“1962”——它作为基期点进入回归与图形（系数为 0），使基期 1962 也能显示在图中。

In [ ]:
%%stata
gen p=1
label var p "1962"


事件研究回归：因变量 ln_ads_p4_cst 是广告价格对数；ib1962.year##ib0.national 生成年份与全国性报纸的全交互，以 1962 年为基期、地方报纸（national=0）为基准组，于是每个“年份×national”交互系数就是该年处理组相对基期、相对对照组的差异（即每年的 DID 系数）。absorb(year id_news) 吸收年份与报纸个体固定效应，标准误聚类到报纸层面。

In [ ]:
%%stata
reghdfe ln_ads_p4_cst p ib1962.year##ib0.national if year>=1962, a(year id_news) cluster(id_news)


coefplot 画出各年的交互系数：keep(p *.year#1.national) 保留基期点与交互项，rename(...regex) 用正则表达式把系数名中的年份数字提取为横轴标签；vertical 竖排系数、yline(0) 为零参考线、xline(5.5) 在政策前后之间画分隔虚线，levels(90) 绘制 90% 置信区间，omitted 标记被省略的基期。

In [ ]:
%%stata
coefplot, omitted vertical label keep(p *.year#1.national) yline(0) levels(90) xline(5.5,lcolor(black) lpattern(dash)) rename((\d+).year#1.national=\1, regex)
